# 🚗 MY CAR AI — 나만의 자동차 관리 챗봇 (Gradio + OpenAI + SQLite)

**현장 데이터 디지털화 과정 · Day05 코랩 실습 예시**

내 차를 등록하고, 정비 이력을 쌓고, 소모품 교체 시기를 확인하고, AI 에게 자동차 관리를 묻는 챗봇입니다.
**계산은 파이썬, 말은 AI** — 교체 시기 · 비용 합계 · 지난 km 는 파이썬이 계산하고, OpenAI 는 그 결과를 읽어 쉬운 말로 답합니다.

### 시작하는 방법
1. 코랩 왼쪽 🔑 **보안 비밀**에 이름 `OPENAI_API_KEY` 로 키를 넣고 **노트북 액세스**를 켭니다.
2. 메뉴 **런타임 → 모두 실행**. ② 셀에서 구글 계정 허용 창이 뜨면 허용합니다(데이터를 드라이브에 저장).
3. ④ 셀 아래 화면(또는 `https://….gradio.live`)의 **시작하기** 탭에 있는 1~6 순서대로 따라 합니다.

## 요청서를 어떻게 반영했나

| 요청 | 이 노트북에서 |
|---|---|
| 차량 여러 대 등록 · 수정 · 삭제 | 「내 차량 관리」 탭 · 맨 위 「관리할 차량」으로 고르기 |
| 연료별 해당 없는 소모품 제외 | 전기차: 엔진오일 · 에어클리너 · 변속기 오일 · 점화플러그 제외 / 디젤: 점화플러그 제외 |
| 마지막 교체 · 다음 교체 날짜와 km · 상태 | 파이썬 계산. km 와 기간 중 **먼저 닿는 기준**, 🟢 정상 · 🟠 교체 예정(1,000km · 30일 안) · 🔴 경과 |
| 정비 이력 표 · 날짜/항목 검색 · 수정 · 삭제 | 「정비 이력」 탭 (날짜는 `2025` 또는 `2025-03` 처럼 앞부분으로 찾기) |
| 대시보드 | 「시작하기」 탭 — 주행거리 · 경과/예정 소모품 · 최근 정비 · 올해 비용 |
| AI 상담 · 근거 · 「확실하지 않은 정보」 · 「AI의 의견」 · 안전 안내 | 파이썬이 계산한 사실 표를 AI 에게 주고, 답을 칸별로 나눠 표시 |
| 후속 질문 단추 1~3개(누르면 바로 답 · 반복) | 「이어서 물어보기」 단추 |
| SQLite + 구글 드라이브 저장 | `내 드라이브/MYCAR_AI/mycar.db` |
| API 키는 코드 · DB 에 저장 안 함 | 보안 비밀에서 읽음(또는 화면 입력칸 — 저장하지 않음) |
| 주황색 디자인 · 탭 5개 | 시작하기 · 내 차량 관리 · 소모품 관리 · 정비 이력 · AI 자동차 상담 |

**요청과 다르게 한 것 · 못 한 것과 이유**
| 요청 | 이렇게 했습니다 | 이유 |
|---|---|---|
| 제조사 공식 교체 주기 적용 | 코드에 숫자를 넣지 않았습니다. **취급설명서에서 확인한 주기와 쪽 번호(또는 URL)** 를 「소모품 관리」 표에 넣으면 그 값으로 계산하고, 비어 있으면 **⚪ 제조사 기준 확인 필요** | 주기는 차종 · 연식 · 엔진 · 운행 조건(가혹 조건)마다 달라 일반 숫자를 넣으면 틀린 값을 확정하게 됩니다 |
| 공식 매뉴얼을 근거로 답 | 저장된 「근거」 칸을 답에 그대로 밝히고, ☑ 「웹에서 제조사 공식 자료 찾아보기」를 켜면 OpenAI 웹 검색으로 찾은 URL 을 붙입니다 | 매뉴얼 원문(PDF)을 프로그램이 직접 읽지는 않습니다 |
| 다음 교체 예상 날짜(km 기준) | **연간 예상 주행거리**를 넣은 경우에만 어림합니다 | 하루에 몇 km 달리는지 모르면 날짜를 계산할 수 없습니다 |
| 타이어 · 브레이크 패드 | 주기를 비워 두면 「확인 필요」로 남습니다 | 보통 거리보다 **마모 상태 점검**으로 교체를 정하는 항목입니다 |
| 전기차 감속기 오일 등 목록 밖 항목 | 「기타 정비」로 기록합니다 | 요청서의 10개 항목만 상태를 계산합니다 |

> 드라이브 연결(②)을 건너뛰면 데이터가 `/content/mycar.db` 에 저장되어 **런타임이 끝나면 사라집니다.** 화면 위쪽 「데이터 파일」 줄에서 저장 위치를 확인하세요.

## ① 설치 (1분쯤)

In [ ]:
!pip install -q -U gradio openai pandas
import gradio, openai
print("Gradio", gradio.__version__, "· openai", openai.__version__)

## ② 구글 드라이브 연결 — 데이터가 사라지지 않게
허용 창이 뜨면 수업에서 쓰는 구글 계정으로 허용하세요. `내 드라이브/MYCAR_AI/mycar.db` 에 저장합니다.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/MYCAR_AI", exist_ok=True)
os.environ["MYCAR_DB"] = "/content/drive/MyDrive/MYCAR_AI/mycar.db"
print("데이터 파일:", os.environ["MYCAR_DB"], "(이미 있으면 이어서 씁니다)" if os.path.exists(os.environ["MYCAR_DB"]) else "(처음 만듭니다)")

## ③ API 키 불러오기 (보안 비밀 `OPENAI_API_KEY`)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ 보안 비밀에서 API 키를 읽었습니다.")
except Exception:
    print("보안 비밀에 키가 없습니다. 챗봇 화면 위쪽 「API 키」 칸에 붙여 넣으세요.")

## ④ MY CAR AI 실행
아래에 화면이 뜹니다. 멈추려면 ■(중지), 코드를 고쳤으면 이 셀만 다시 실행하세요. (DB · 계산 · 정비 이력 · AI 상담 · 후속 질문 · 화면이 이 셀 하나에 순서대로 들어 있습니다)

In [ ]:
# ============================================================
# MY CAR AI — 나만의 자동차 관리 챗봇 (Gradio + OpenAI + SQLite)
#  역할 나누기
#   - 파이썬: 차량 · 정비 이력 저장(SQLite), 소모품 교체 시기 계산, 비용 합계
#   - OpenAI: 질문 해석과 쉬운 말 답변, (선택) 제조사 자료 웹 검색
#  교체 주기는 차종마다 달라서 코드에 숫자를 박아 두지 않습니다.
#  취급설명서에서 확인한 값을 사용자가 넣고, 넣지 않은 항목은 「제조사 기준 확인 필요」로 표시합니다.
# ============================================================
import os, re, json, sqlite3, inspect, datetime as dt
import time
os.environ["TZ"] = "Asia/Seoul"   # 코랩 시계는 UTC — 「오늘」 · 시각을 한국 기준으로
time.tzset()
import pandas as pd
import gradio as gr
from openai import OpenAI

DB_PATH = os.environ.get("MYCAR_DB", "/content/mycar.db")
MODEL_DEFAULT = "gpt-4.1-mini"
SOON_KM, SOON_DAYS = 1000, 30          # 이만큼 남으면 「교체 예정」

FUELS = ["가솔린", "디젤", "LPG", "하이브리드", "전기차"]
ITEMS = ["엔진오일 및 오일필터", "에어클리너", "에어컨 필터", "브레이크액", "브레이크 패드",
         "타이어", "냉각수", "변속기 오일", "배터리", "점화플러그"]
# 연료마다 해당하지 않는 항목 (파이썬 규칙)
NOT_FOR = {
    "전기차": {"엔진오일 및 오일필터", "에어클리너", "변속기 오일", "점화플러그"},
    "디젤": {"점화플러그"},          # 디젤은 점화플러그 대신 예열플러그를 씁니다
}
SERVICE_ITEMS = ITEMS + ["기타 정비"]


# ---------- 1. 데이터베이스 ----------
def db():
    os.makedirs(os.path.dirname(DB_PATH) or ".", exist_ok=True)
    con = sqlite3.connect(DB_PATH)
    con.execute("PRAGMA foreign_keys = ON")
    return con

def init_db():
    with db() as con:
        con.executescript("""
        CREATE TABLE IF NOT EXISTS cars(
            id INTEGER PRIMARY KEY AUTOINCREMENT, maker TEXT, model TEXT, year INTEGER, fuel TEXT,
            odo INTEGER, odo_date TEXT, purchase_date TEXT, annual_km INTEGER);
        CREATE TABLE IF NOT EXISTS services(
            id INTEGER PRIMARY KEY AUTOINCREMENT, car_id INTEGER REFERENCES cars(id) ON DELETE CASCADE,
            date TEXT, odo INTEGER, item TEXT, cost INTEGER, shop TEXT, memo TEXT);
        CREATE TABLE IF NOT EXISTS intervals(
            car_id INTEGER REFERENCES cars(id) ON DELETE CASCADE, item TEXT,
            km INTEGER, months INTEGER, source TEXT, PRIMARY KEY(car_id, item));
        """)

def q(sql, args=()):
    with db() as con:
        con.row_factory = sqlite3.Row
        return [dict(r) for r in con.execute(sql, args).fetchall()]

def x(sql, args=()):
    with db() as con:
        cur = con.execute(sql, args)
        return cur.lastrowid


# ---------- 2. 입력값 다듬기 ----------
def to_int(v, name, required=False):
    s = re.sub(r"[^\d]", "", str(v or ""))
    if not s:
        if required:
            raise gr.Error(f"「{name}」을(를) 숫자로 넣어 주세요.")
        return None
    return int(s)

def to_date(v, name, required=False):
    s = str(v or "").strip().replace(".", "-").replace("/", "-")
    if not s:
        if required:
            raise gr.Error(f"「{name}」을(를) 2025-03-15 처럼 넣어 주세요.")
        return None
    try:
        return dt.date.fromisoformat(s).isoformat()
    except ValueError:
        raise gr.Error(f"「{name}」 날짜 모양이 맞지 않습니다: {s} (예: 2025-03-15)")

def car_label(c):
    return f"{c['id']} · {c['maker']} {c['model']} ({c['year']}년 · {c['fuel']})"

def car_choices():
    return [car_label(c) for c in q("SELECT * FROM cars ORDER BY id")]

def car_id_of(label):
    if not label:
        return None
    return int(str(label).split(" · ")[0])

def get_car(label):
    cid = car_id_of(label)
    rows = q("SELECT * FROM cars WHERE id=?", (cid,)) if cid else []
    return rows[0] if rows else None

def need_car(label):
    c = get_car(label)
    if not c:
        raise gr.Error("먼저 위에서 차량을 고르거나 「내 차량 관리」에서 차량을 등록해 주세요.")
    return c


# ---------- 3. 소모품 교체 시기 계산 (파이썬) ----------
def add_months(d, m):
    y, mo = d.year + (d.month - 1 + m) // 12, (d.month - 1 + m) % 12 + 1
    day = min(d.day, [31, 29 if y % 4 == 0 and (y % 100 or y % 400 == 0) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][mo - 1])
    return dt.date(y, mo, day)

def items_for(car):
    return [i for i in ITEMS if i not in NOT_FOR.get(car["fuel"], set())]

def consumable_status(car, today=None):
    """소모품마다 마지막 교체 · 다음 교체 · 상태를 계산합니다"""
    today = today or dt.date.today()
    iv = {r["item"]: r for r in q("SELECT * FROM intervals WHERE car_id=?", (car["id"],))}
    rows = []
    for item in items_for(car):
        last = q("SELECT date, odo FROM services WHERE car_id=? AND item=? ORDER BY date DESC, odo DESC LIMIT 1",
                 (car["id"], item))
        last = last[0] if last else None
        rule = iv.get(item) or {}
        km, months, src = rule.get("km"), rule.get("months"), rule.get("source") or ""
        r = {"소모품": item, "마지막 교체일": last["date"] if last else "-",
             "마지막 교체 km": f"{last['odo']:,}" if last and last["odo"] is not None else "-",
             "교체 주기": " / ".join(s for s in [f"{km:,}km" if km else "", f"{months}개월" if months else ""] if s) or "-",
             "근거": src or "-", "다음 교체 예상일": "-", "다음 교체 예상 km": "-", "상태": ""}
        if not (km or months):
            r["상태"] = "⚪ 제조사 기준 확인 필요"
        elif not last:
            r["상태"] = "⚪ 교체 기록 없음"
        else:
            dates, overdue, soon = [], False, False
            if km and last["odo"] is not None:
                nk = last["odo"] + km
                r["다음 교체 예상 km"] = f"{nk:,}"
                left = nk - (car["odo"] or 0)
                overdue |= left <= 0
                soon |= 0 < left <= SOON_KM
                if car.get("annual_km") and left > 0:      # 연간 주행거리로 그 km 에 닿는 날짜를 어림
                    dates.append(today + dt.timedelta(days=int(left / car["annual_km"] * 365)))
            if months:
                nd = add_months(dt.date.fromisoformat(last["date"]), months)
                dates.append(nd)
                overdue |= nd <= today
                soon |= today < nd <= today + dt.timedelta(days=SOON_DAYS)
            if dates:
                r["다음 교체 예상일"] = min(dates).isoformat()      # 먼저 닿는 기준
            r["상태"] = "🔴 교체 시기 경과" if overdue else "🟠 교체 예정" if soon else "🟢 정상"
        r["_km_since"] = (car["odo"] - last["odo"]) if last and last["odo"] is not None and car["odo"] else None
        rows.append(r)
    return rows

def status_df(car):
    return pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in consumable_status(car)])


# ---------- 4. 차량 등록 · 수정 · 삭제 ----------
def save_car(sel, mode, maker, model, year, fuel, odo, purchase, annual):
    if not (maker or "").strip() or not (model or "").strip():
        raise gr.Error("제조사와 모델명을 넣어 주세요. (예: 현대 · 아반떼)")
    if fuel not in FUELS:
        raise gr.Error("연료 종류를 골라 주세요.")
    vals = (maker.strip(), model.strip(), to_int(year, "연식", True), fuel, to_int(odo, "현재 주행거리", True),
            dt.date.today().isoformat(), to_date(purchase, "구매일"), to_int(annual, "연간 예상 주행거리"))
    if mode == "새 차량으로 등록":
        cid = x("INSERT INTO cars(maker,model,year,fuel,odo,odo_date,purchase_date,annual_km) VALUES(?,?,?,?,?,?,?,?)", vals)
        msg = "✅ 새 차량을 등록했습니다."
    else:
        c = need_car(sel)
        cid = c["id"]
        x("UPDATE cars SET maker=?,model=?,year=?,fuel=?,odo=?,odo_date=?,purchase_date=?,annual_km=? WHERE id=?", vals + (cid,))
        msg = "✅ 차량 정보를 고쳤습니다."
    label = car_label(q("SELECT * FROM cars WHERE id=?", (cid,))[0])
    return gr.update(choices=car_choices(), value=label), msg

def load_car(sel):
    c = get_car(sel)
    if not c:
        return [gr.update()] * 7
    return (c["maker"], c["model"], c["year"], c["fuel"], c["odo"], c["purchase_date"] or "", c["annual_km"] or "")

def delete_car(sel, confirm):
    c = need_car(sel)
    if not confirm:
        raise gr.Error("지우려면 「정말 지웁니다」에 체크해 주세요. 정비 이력도 함께 지워집니다.")
    x("DELETE FROM cars WHERE id=?", (c["id"],))
    ch = car_choices()
    return gr.update(choices=ch, value=ch[0] if ch else None), f"🗑️ {c['maker']} {c['model']} 을(를) 지웠습니다.", False

def update_odo(sel, odo):
    c = need_car(sel)
    v = to_int(odo, "현재 주행거리", True)
    if v < (c["odo"] or 0):
        raise gr.Error(f"주행거리가 전보다 작습니다(전: {c['odo']:,}km). 숫자를 확인해 주세요.")
    x("UPDATE cars SET odo=?, odo_date=? WHERE id=?", (v, dt.date.today().isoformat(), c["id"]))
    return f"✅ 현재 주행거리를 {v:,}km 로 바꿨습니다."


# ---------- 5. 교체 주기(취급설명서 값) ----------
def interval_table(sel):
    c = get_car(sel)
    if not c:
        return pd.DataFrame(columns=["소모품", "교체 주기(km)", "교체 주기(개월)", "근거(취급설명서 쪽 · URL)"])
    iv = {r["item"]: r for r in q("SELECT * FROM intervals WHERE car_id=?", (c["id"],))}
    return pd.DataFrame([{"소모품": i, "교체 주기(km)": (iv.get(i) or {}).get("km") or "",
                          "교체 주기(개월)": (iv.get(i) or {}).get("months") or "",
                          "근거(취급설명서 쪽 · URL)": (iv.get(i) or {}).get("source") or ""} for i in items_for(c)])

def save_intervals(sel, table):
    c = need_car(sel)
    df = table if isinstance(table, pd.DataFrame) else pd.DataFrame(table)
    n = 0
    for _, r in df.iterrows():
        item = str(r.iloc[0]).strip()
        if item not in ITEMS:
            continue
        km, mo, src = to_int(r.iloc[1], "교체 주기(km)"), to_int(r.iloc[2], "교체 주기(개월)"), str(r.iloc[3] or "").strip()
        if (km or mo) and not src:
            raise gr.Error(f"「{item}」의 근거(취급설명서 몇 쪽인지, 또는 URL)를 적어 주세요.")
        x("INSERT OR REPLACE INTO intervals(car_id,item,km,months,source) VALUES(?,?,?,?,?)", (c["id"], item, km, mo, src))
        n += 1 if (km or mo) else 0
    return f"✅ 교체 주기를 저장했습니다(값이 있는 항목 {n}개).", status_df(c)


# ---------- 6. 정비 이력 ----------
def service_df(sel, year="", item="전체"):
    c = get_car(sel)
    cols = ["ID", "날짜", "주행거리(km)", "정비 항목", "비용(원)", "정비소", "메모"]
    if not c:
        return pd.DataFrame(columns=cols)
    sql, args = "SELECT * FROM services WHERE car_id=?", [c["id"]]
    if (year or "").strip():
        sql += " AND date LIKE ?"
        args.append(year.strip() + "%")        # 2025 또는 2025-03 처럼 앞부분으로 찾기
    if item and item != "전체":
        sql += " AND item=?"
        args.append(item)
    rows = q(sql + " ORDER BY date DESC, odo DESC", args)
    return pd.DataFrame([[r["id"], r["date"], f"{r['odo']:,}" if r["odo"] is not None else "", r["item"],
                          f"{r['cost']:,}" if r["cost"] is not None else "", r["shop"] or "", r["memo"] or ""]
                         for r in rows], columns=cols)

def service_choices(sel):
    c = get_car(sel)
    if not c:
        return []
    return [f"{r['id']} · {r['date']} · {r['item']}" for r in
            q("SELECT id,date,item FROM services WHERE car_id=? ORDER BY date DESC", (c["id"],))]

def save_service(sel, pick, date, odo, item, cost, shop, memo):
    c = need_car(sel)
    if item not in SERVICE_ITEMS:
        raise gr.Error("정비 항목을 골라 주세요.")
    vals = (to_date(date, "정비 날짜", True), to_int(odo, "정비 당시 주행거리"), item,
            to_int(cost, "정비 비용"), (shop or "").strip(), (memo or "").strip())
    if pick:                                   # 고를 이력이 있으면 고치기
        x("UPDATE services SET date=?,odo=?,item=?,cost=?,shop=?,memo=? WHERE id=? AND car_id=?",
          vals + (int(pick.split(" · ")[0]), c["id"]))
        msg = "✅ 정비 이력을 고쳤습니다."
    else:
        x("INSERT INTO services(car_id,date,odo,item,cost,shop,memo) VALUES(?,?,?,?,?,?,?)", (c["id"],) + vals)
        msg = "✅ 정비 이력을 추가했습니다."
    # 정비 당시 주행거리가 현재 주행거리보다 크면 현재 주행거리를 올립니다
    if vals[1] and vals[1] > (c["odo"] or 0):
        x("UPDATE cars SET odo=?, odo_date=? WHERE id=?", (vals[1], vals[0], c["id"]))
        msg += f" (현재 주행거리도 {vals[1]:,}km 로 올렸습니다)"
    return msg, service_df(sel), gr.update(choices=service_choices(sel), value=None)

def load_service(pick):
    if not pick:
        return ["", "", None, "", "", ""]
    r = q("SELECT * FROM services WHERE id=?", (int(pick.split(" · ")[0]),))[0]
    return [r["date"], r["odo"] or "", r["item"], r["cost"] or "", r["shop"] or "", r["memo"] or ""]

def delete_service(sel, pick):
    c = need_car(sel)
    if not pick:
        raise gr.Error("지울 정비 이력을 먼저 골라 주세요.")
    x("DELETE FROM services WHERE id=? AND car_id=?", (int(pick.split(" · ")[0]), c["id"]))
    return "🗑️ 정비 이력을 지웠습니다.", service_df(sel), gr.update(choices=service_choices(sel), value=None)


# ---------- 7. 대시보드 ----------
def cost_by_year(car):
    return {r["y"]: r["s"] for r in q("SELECT substr(date,1,4) y, SUM(COALESCE(cost,0)) s FROM services WHERE car_id=? GROUP BY y ORDER BY y", (car["id"],))}

def dashboard(sel):
    c = get_car(sel)
    if not c:
        return ("### 아직 등록한 차량이 없습니다\n「내 차량 관리」 탭에서 차량을 먼저 등록해 주세요.", pd.DataFrame())
    st = consumable_status(c)
    over = [r["소모품"] for r in st if r["상태"].startswith("🔴")]
    soon = [r["소모품"] for r in st if r["상태"].startswith("🟠")]
    unknown = [r["소모품"] for r in st if r["상태"].startswith("⚪")]
    year = str(dt.date.today().year)
    recent = q("SELECT date,item,cost FROM services WHERE car_id=? ORDER BY date DESC LIMIT 3", (c["id"],))
    md = f"""### 🚗 {c['maker']} {c['model']} · {c['year']}년 · {c['fuel']}
<div class="cards">
<div class="card"><b>현재 주행거리</b><span>{c['odo']:,} km</span><small>{c['odo_date']} 기준</small></div>
<div class="card red"><b>🔴 교체 시기 경과</b><span>{len(over)}개</span><small>{', '.join(over) or '없음'}</small></div>
<div class="card orange"><b>🟠 교체 예정</b><span>{len(soon)}개</span><small>{', '.join(soon) or '없음'}</small></div>
<div class="card"><b>{year}년 정비 비용</b><span>{cost_by_year(c).get(year, 0):,}원</span><small>정비 이력 합계</small></div>
</div>

**최근 정비**: {' · '.join(f"{r['date']} {r['item']}" + (f" ({r['cost']:,}원)" if r['cost'] else '') for r in recent) or '아직 없음'}
""" + (f"\n\n⚪ **기준 확인 필요 {len(unknown)}개** — 「소모품 관리」에서 취급설명서의 교체 주기를 넣으면 계산합니다." if unknown else "")
    return md, status_df(c)


# ---------- 8. AI 상담 ----------
SYSTEM_PROMPT = """너는 자동차 정비 전문가이자 친절한 자동차 관리 상담사야.
[지켜야 할 것]
1. [내 차 정보]·[소모품 상태]·[정비 이력]·[연도별 비용]은 프로그램이 계산한 사실이야. 숫자(날짜 · km · 비용)는 여기 있는 값만 쓰고 직접 다시 계산하거나 지어내지 마.
2. 교체 주기의 근거는 [소모품 상태]의 「근거」 칸(취급설명서 쪽 · URL)을 그대로 밝혀. 근거가 없는 일반 지식은 uncertain 에 넣어 「확실하지 않은 정보」로 구분해.
3. 추천 · 추정은 ai_opinion 에 따로 적어 「AI의 의견」으로 구분해.
4. 고장 원인을 단정하지 말고 「가능한 원인」과 「점검 방법」을 나눠 설명해.
5. 브레이크 · 조향 · 타이어 손상 · 냉각수 경고 · 연기 · 타는 냄새처럼 안전에 영향을 주는 증상이면 safety 에 운행을 멈춰야 하는지와 전문 정비소 점검을 꼭 적어.
6. 답은 한국어로, 운전자가 바로 이해할 수 있게 짧고 쉽게 써.
7. 이어서 물어볼 만한 질문 1~3개를 followups 에 만들어(이 차와 이력에 맞게).
반드시 아래 JSON 하나로만 답해:
{"answer": "답(마크다운)", "sources": ["참고한 근거: 취급설명서 쪽 · URL · 정비 이력 ID 등"],
 "uncertain": ["확실하지 않은 정보"], "ai_opinion": "AI의 의견(없으면 빈 글자)", "safety": "안전 안내(없으면 빈 글자)",
 "followups": ["후속 질문1", "후속 질문2"]}"""

def facts_text(c):
    st = consumable_status(c)
    lines = [f"[내 차 정보] {c['maker']} {c['model']} · {c['year']}년식 · {c['fuel']} · 현재 {c['odo']:,}km({c['odo_date']} 기준)"
             f" · 구매일 {c['purchase_date'] or '모름'} · 연간 예상 주행 {c['annual_km'] or '모름'}km · 오늘 {dt.date.today()}",
             "[소모품 상태]"]
    for r in st:
        since = f" · 마지막 교체 뒤 {r['_km_since']:,}km 주행" if r["_km_since"] is not None else ""
        lines.append(f"- {r['소모품']}: 상태 {r['상태']} · 마지막 {r['마지막 교체일']}/{r['마지막 교체 km']}km · 주기 {r['교체 주기']}"
                     f" · 근거 {r['근거']} · 다음 {r['다음 교체 예상일']}/{r['다음 교체 예상 km']}km{since}")
    lines.append("[정비 이력] (ID · 날짜 · km · 항목 · 비용 · 정비소 · 메모)")
    for r in q("SELECT * FROM services WHERE car_id=? ORDER BY date DESC LIMIT 40", (c["id"],)):
        lines.append(f"- #{r['id']} {r['date']} · {r['odo'] or '-'}km · {r['item']} · {r['cost'] or 0}원 · {r['shop'] or '-'} · {r['memo'] or ''}")
    lines.append("[연도별 비용] " + (", ".join(f"{y}년 {s:,}원" for y, s in cost_by_year(c).items()) or "기록 없음"))
    return "\n".join(lines)

def get_client(key_box):
    key = (key_box or "").strip() or os.environ.get("OPENAI_API_KEY", "")
    if not key:
        raise gr.Error("OpenAI API 키가 없습니다. 코랩 보안 비밀 OPENAI_API_KEY 에 넣거나 위 「API 키」 칸에 붙여 넣어 주세요.")
    return OpenAI(api_key=key)

def to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content or "")

def parse_json(raw):
    raw = (raw or "").strip()
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        return json.loads(m.group(0) if m else raw)
    except Exception:
        return {"answer": raw, "followups": []}

def friendly(e):
    s = str(e)
    if "invalid_api_key" in s or "Incorrect API key" in s:
        return "API 키가 맞지 않습니다. 키를 다시 확인해 주세요."
    if "insufficient_quota" in s:
        return "OpenAI 계정의 사용 한도(크레딧)가 없습니다. platform.openai.com 의 Billing 을 확인해 주세요."
    if "rate_limit" in s or "429" in s:
        return "요청이 너무 빨라 잠시 막혔습니다. 1분 뒤 다시 눌러 주세요."
    if "model_not_found" in s or "does not exist" in s:
        return "이 키로는 고른 모델을 쓸 수 없습니다. 위 「모델」에서 다른 모델을 골라 보세요."
    return "오류가 났습니다: " + s[:300]

def ask(sel, key_box, model, use_web, question, history):
    question = (question or "").strip()
    if not question:
        raise gr.Error("질문을 적어 주세요.")
    c = need_car(sel)
    history = history or []
    client = get_client(key_box)
    facts = facts_text(c)
    msgs = [{"role": "system", "content": SYSTEM_PROMPT + "\n\n" + facts}]
    for m in history[-6:]:                          # 앞 대화 3번까지 기억
        msgs.append({"role": m["role"], "content": to_text(m["content"])[:2500]})
    msgs.append({"role": "user", "content": question})
    try:
        if use_web:
            # 웹 검색: 제조사 공식 자료(취급설명서 · 공식 홈페이지)를 찾아 URL 을 근거로 붙입니다
            prompt = "\n\n".join(f"[{m['role']}]\n{m['content']}" for m in msgs) + \
                     "\n\n제조사 공식 자료(취급설명서 · 공식 홈페이지)를 우선 찾아 보고, 찾은 URL 을 sources 에 넣어. JSON 하나로만 답해."
            try:
                r = client.responses.create(model=model, input=prompt, tools=[{"type": "web_search_preview"}])
                data = parse_json(r.output_text)
            except Exception as e:
                if "web_search" not in str(e) and "tool" not in str(e):
                    raise
                data = None
            if data is None:
                r = client.chat.completions.create(model=model, messages=msgs, response_format={"type": "json_object"})
                data = parse_json(r.choices[0].message.content)
                data.setdefault("uncertain", []).append("웹 검색을 쓸 수 없는 모델이라 검색 없이 답했습니다.")
        else:
            r = client.chat.completions.create(model=model, messages=msgs, response_format={"type": "json_object"})
            data = parse_json(r.choices[0].message.content)
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(friendly(e))

    text = (data.get("answer") or "").strip()
    if data.get("safety"):
        text = f"> ⚠️ **안전 안내** — {data['safety']}\n\n" + text
    if data.get("ai_opinion"):
        text += f"\n\n**【AI의 의견】** {data['ai_opinion']}"
    unc = [u for u in (data.get("uncertain") or []) if str(u).strip()]
    if unc:
        text += "\n\n**【확실하지 않은 정보】**\n" + "\n".join(f"- {u}" for u in unc)
    src = [s for s in (data.get("sources") or []) if str(s).strip()]
    if src:
        text += "\n\n**참고한 근거**\n" + "\n".join(f"- {s}" for s in src)
    fups = [f for f in (data.get("followups") or []) if isinstance(f, str) and f.strip()][:3]
    history = history + [{"role": "user", "content": question}, {"role": "assistant", "content": text}]
    btns = [gr.update(value=fups[i], visible=True) if i < len(fups) else gr.update(visible=False) for i in range(3)]
    return ("", history, *btns)


# ---------- 9. 화면 ----------
HOWTO = """## 사용 방법 (처음 한 번만 순서대로)
| 순서 | 할 일 | 입력 예시 |
|---|---|---|
| 1 | **OpenAI API 키** — 코랩 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY` 로 넣기 (또는 위 「API 키」 칸) | `sk-...` |
| 2 | **내 차량 등록** — 「내 차량 관리」 탭 | 현대 · 아반떼 · 2021 · 가솔린 · 45200 |
| 3 | **현재 주행거리** — 계기판 숫자를 넣고 「주행거리 바꾸기」 | 45200 |
| 4 | **정비 이력 등록** — 「정비 이력」 탭 | 2025-03-15 · 40100 · 엔진오일 및 오일필터 · 78000 · 블루핸즈 |
| 5 | **교체 주기 넣고 상태 보기** — 「소모품 관리」 탭에 **취급설명서**의 주기와 쪽 번호 | 엔진오일 15000km · 12개월 · 취급설명서 8-5쪽 |
| 6 | **AI 상담** — 「AI 자동차 상담」 탭 | 「지금 교체가 필요한 소모품은?」 |

> 교체 주기는 차종 · 운행 조건마다 다르므로 프로그램이 숫자를 정하지 않습니다. 넣지 않은 항목은 **⚪ 제조사 기준 확인 필요**로 보입니다.
> 상태 표시: 🟢 정상 · 🟠 교체 예정(1,000km 또는 30일 안) · 🔴 교체 시기 경과 · ⚪ 기준 또는 기록 없음"""

CSS = """
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin:8px 0}
.card{border:1px solid #fed7aa;border-radius:12px;padding:12px 14px;background:#fff7ed}
.card b{display:block;font-size:.9em;color:#9a3412}.card span{display:block;font-size:1.5em;font-weight:700;margin:2px 0}
.card small{color:#57534e;word-break:keep-all}.card.red{background:#fef2f2;border-color:#fecaca}.card.orange{background:#fff7ed;border-color:#fdba74}
"""
THEME = gr.themes.Soft(primary_hue="orange", secondary_hue="amber")

chat_kwargs = {"label": "AI 자동차 상담", "height": 460}
if "type" in inspect.signature(gr.Chatbot.__init__).parameters:     # Gradio 4~5
    chat_kwargs["type"] = "messages"
blocks_kwargs = {"title": "MY CAR AI"}
launch_kwargs = {"share": True, "debug": True}
if "theme" in inspect.signature(gr.Blocks.launch).parameters:        # Gradio 6: 테마 · CSS 는 launch 에
    launch_kwargs.update(theme=THEME, css=CSS)
else:
    blocks_kwargs.update(theme=THEME, css=CSS)

init_db()
with gr.Blocks(**blocks_kwargs) as demo:
    gr.Markdown("# 🚗 MY CAR AI — 나만의 자동차 관리 챗봇")
    with gr.Row():
        car_sel = gr.Dropdown(car_choices(), label="관리할 차량", scale=3,
                              value=(car_choices() or [None])[0])
        key_box = gr.Textbox(label="API 키 (보안 비밀에 넣었다면 비워 두기)", type="password", scale=2)
        model = gr.Dropdown(["gpt-4.1-mini", "gpt-4o-mini", "gpt-4.1"], value=MODEL_DEFAULT, label="모델",
                            allow_custom_value=True, scale=1)
    gr.Markdown(f"<small>데이터 파일: `{DB_PATH}`" + (" — 구글 드라이브에 저장되어 런타임을 다시 시작해도 남습니다." if DB_PATH.startswith("/content/drive")
                else " — ⚠️ 코랩 임시 폴더라 런타임이 끝나면 사라집니다. 노트북의 ② 드라이브 연결 셀을 실행하세요.") + "</small>")

    with gr.Tabs():
        with gr.Tab("시작하기"):
            dash_md = gr.Markdown()
            dash_tbl = gr.Dataframe(label="소모품 상태", interactive=False, wrap=True)
            refresh = gr.Button("새로 고침", size="sm")
            with gr.Accordion("사용 방법", open=True):
                gr.Markdown(HOWTO)

        with gr.Tab("내 차량 관리"):
            car_mode = gr.Radio(["새 차량으로 등록", "고른 차량 고치기"], value="새 차량으로 등록", label="할 일")
            with gr.Row():
                f_maker = gr.Textbox(label="제조사", placeholder="현대")
                f_model = gr.Textbox(label="차량 모델명", placeholder="아반떼")
                f_year = gr.Number(label="연식", precision=0, value=None)
                f_fuel = gr.Dropdown(FUELS, label="연료 종류")
            with gr.Row():
                f_odo = gr.Number(label="현재 주행거리(km)", precision=0)
                f_purchase = gr.Textbox(label="구매일(선택)", placeholder="2021-05-20")
                f_annual = gr.Number(label="연간 예상 주행거리(km, 선택)", precision=0)
            with gr.Row():
                save_car_btn = gr.Button("저장", variant="primary")
                load_car_btn = gr.Button("고른 차량 불러오기")
            car_msg = gr.Markdown()
            gr.Markdown("---\n**현재 주행거리만 바꾸기**")
            with gr.Row():
                new_odo = gr.Number(label="계기판 주행거리(km)", precision=0)
                odo_btn = gr.Button("주행거리 바꾸기")
            with gr.Accordion("차량 지우기", open=False):
                del_ok = gr.Checkbox(label="정말 지웁니다 (정비 이력도 함께 지워짐)")
                del_car_btn = gr.Button("고른 차량 지우기", variant="stop")

        with gr.Tab("소모품 관리"):
            gr.Markdown("**교체 주기**는 차량 **취급설명서**의 「정기 점검 · 교체」 표에서 찾아 넣고, 근거 칸에 쪽 번호나 URL 을 적어 주세요. "
                        "연료 종류에 해당하지 않는 항목(예: 전기차의 엔진오일)은 자동으로 뺐습니다.")
            iv_tbl = gr.Dataframe(label="교체 주기 (표 안을 눌러 고친 뒤 「교체 주기 저장」)", interactive=True, wrap=True)
            iv_btn = gr.Button("교체 주기 저장", variant="primary")
            iv_msg = gr.Markdown()
            st_tbl = gr.Dataframe(label="소모품 상태 (파이썬 계산)", interactive=False, wrap=True)

        with gr.Tab("정비 이력"):
            with gr.Row():
                s_pick = gr.Dropdown([], label="고칠 · 지울 이력 (비워 두면 새로 추가)", scale=3)
                s_load = gr.Button("불러오기", scale=1)
            with gr.Row():
                s_date = gr.Textbox(label="정비 날짜", placeholder="2025-03-15")
                s_odo = gr.Number(label="정비 당시 주행거리(km)", precision=0)
                s_item = gr.Dropdown(SERVICE_ITEMS, label="정비 항목")
                s_cost = gr.Number(label="정비 비용(원)", precision=0)
            with gr.Row():
                s_shop = gr.Textbox(label="정비소 이름")
                s_memo = gr.Textbox(label="정비 내용 · 메모", scale=2)
            with gr.Row():
                s_save = gr.Button("저장", variant="primary")
                s_del = gr.Button("고른 이력 지우기", variant="stop")
            s_msg = gr.Markdown()
            with gr.Row():
                f_year_s = gr.Textbox(label="날짜로 찾기", placeholder="2025 또는 2025-03")
                f_item_s = gr.Dropdown(["전체"] + SERVICE_ITEMS, value="전체", label="항목으로 찾기")
                s_find = gr.Button("찾기")
            s_tbl = gr.Dataframe(label="정비 이력", interactive=False, wrap=True)

        with gr.Tab("AI 자동차 상담"):
            use_web = gr.Checkbox(label="웹에서 제조사 공식 자료 찾아보기 (느리지만 URL 근거를 붙임)", value=False)
            chat = gr.Chatbot(**chat_kwargs)
            q_box = gr.Textbox(label="질문", placeholder="예: 지금 교체가 필요한 소모품은 무엇이야?")
            gr.Markdown("**이어서 물어보기** (누르면 바로 답합니다)")
            with gr.Row():
                fbtns = [gr.Button(visible=False, size="sm") for _ in range(3)]
            gr.Examples(["내가 마지막으로 엔진오일을 교체한 게 언제야?", "작년에 자동차 정비에 얼마를 사용했어?",
                         "타이어를 교체한 지 몇 km 지났어?", "브레이크를 밟으면 소리가 나는데 원인이 뭐야?"], q_box)
            clear = gr.Button("대화 지우기", size="sm")

    # ----- 연결 -----
    def refresh_all(sel):
        md, df = dashboard(sel)
        c = get_car(sel)
        return (md, df, interval_table(sel), status_df(c) if c else pd.DataFrame(),
                service_df(sel), gr.update(choices=service_choices(sel), value=None))
    all_outs = [dash_md, dash_tbl, iv_tbl, st_tbl, s_tbl, s_pick]

    demo.load(refresh_all, car_sel, all_outs)
    car_sel.change(refresh_all, car_sel, all_outs)
    refresh.click(refresh_all, car_sel, all_outs)

    save_car_btn.click(save_car, [car_sel, car_mode, f_maker, f_model, f_year, f_fuel, f_odo, f_purchase, f_annual],
                       [car_sel, car_msg]).then(refresh_all, car_sel, all_outs)
    load_car_btn.click(load_car, car_sel, [f_maker, f_model, f_year, f_fuel, f_odo, f_purchase, f_annual]) \
        .then(lambda: "고른 차량 고치기", None, car_mode)
    odo_btn.click(update_odo, [car_sel, new_odo], car_msg).then(refresh_all, car_sel, all_outs)
    del_car_btn.click(delete_car, [car_sel, del_ok], [car_sel, car_msg, del_ok]).then(refresh_all, car_sel, all_outs)

    iv_btn.click(save_intervals, [car_sel, iv_tbl], [iv_msg, st_tbl]).then(refresh_all, car_sel, all_outs)

    s_load.click(load_service, s_pick, [s_date, s_odo, s_item, s_cost, s_shop, s_memo])
    s_save.click(save_service, [car_sel, s_pick, s_date, s_odo, s_item, s_cost, s_shop, s_memo],
                 [s_msg, s_tbl, s_pick]).then(refresh_all, car_sel, all_outs)
    s_del.click(delete_service, [car_sel, s_pick], [s_msg, s_tbl, s_pick]).then(refresh_all, car_sel, all_outs)
    s_find.click(service_df, [car_sel, f_year_s, f_item_s], s_tbl)

    outs = [q_box, chat, *fbtns]
    q_box.submit(ask, [car_sel, key_box, model, use_web, q_box, chat], outs)
    for b in fbtns:                     # 단추 글자(=질문)를 그대로 질문으로 보냅니다
        b.click(ask, [car_sel, key_box, model, use_web, b, chat], outs)
    clear.click(lambda: ([], *[gr.update(visible=False)] * 3), None, [chat, *fbtns])

if __name__ == "__main__":
    demo.launch(**launch_kwargs)

## 오류가 나면

| 증상 | 원인 | 해결 |
|---|---|---|
| 「OpenAI API 키가 없습니다」 | 보안 비밀 이름이 다르거나 노트북 액세스가 꺼짐 | 이름을 정확히 `OPENAI_API_KEY` 로 · 노트북 액세스 켜기 → ③ · ④ 다시 실행 |
| 「사용 한도(크레딧)가 없습니다」 | OpenAI 계정 결제 · 크레딧 없음 | platform.openai.com → Billing 확인 |
| 다음 날 열었더니 차량이 없음 | ② 드라이브 연결을 건너뜀 | ② 를 실행한 뒤 ④ 를 다시 실행 (화면 위 「데이터 파일」이 `/content/drive/...` 인지 확인) |
| 소모품이 모두 ⚪ | 교체 주기를 아직 안 넣음 / 교체 기록이 없음 | 「소모품 관리」에 주기 · 근거 입력, 「정비 이력」에 마지막 교체 기록 |
| 표를 고쳤는데 그대로 | 「교체 주기 저장」을 안 누름 | 칸을 고친 뒤 Enter → 「교체 주기 저장」 |
| `Data incompatible with messages format` | 옛 Gradio 방식(튜플) 코드 | 이 노트북은 Gradio 6 형식(`role` · `content`)으로 되어 있습니다 |
| 웹 검색을 켰는데 「웹 검색을 쓸 수 없는 모델」 | 고른 모델이 웹 검색 도구를 지원하지 않음 | 모델을 `gpt-4.1-mini` 또는 `gpt-4.1` 로 |